# Image Data Augmentation and Classification

This notebook shows a simple **image classification** example with and without augmentation.

We use:
- TensorFlow
- PyTorch
- A/B test: baseline vs augmented training

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 42
tf.keras.utils.set_random_seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# ---------- TensorFlow ----------
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.squeeze()
y_test = y_test.squeeze()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_val = x_train[-5000:]
y_val = y_train[-5000:]
x_train_small = x_train[:20000]
y_train_small = y_train[:20000]

In [ ]:
tf_aug = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

def build_tf_cifar(use_aug=False):
    inputs = keras.Input(shape=(32, 32, 3))
    x = tf_aug(inputs) if use_aug else inputs
    x = layers.Conv2D(32, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    outputs = layers.Dense(10, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

baseline_tf = build_tf_cifar(use_aug=False)
baseline_tf.fit(x_train_small, y_train_small, validation_data=(x_val, y_val), epochs=3, batch_size=128, verbose=0)
base_acc = baseline_tf.evaluate(x_test, y_test, verbose=0)[1]

aug_tf_model = build_tf_cifar(use_aug=True)
aug_tf_model.fit(x_train_small, y_train_small, validation_data=(x_val, y_val), epochs=3, batch_size=128, verbose=0)
aug_acc = aug_tf_model.evaluate(x_test, y_test, verbose=0)[1]

print("TensorFlow baseline accuracy:", round(base_acc, 4))
print("TensorFlow augmented accuracy:", round(aug_acc, 4))

In [ ]:
# ---------- PyTorch ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base_transform = transforms.ToTensor()
aug_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

train_base = datasets.CIFAR10(root="./data", train=True, download=True, transform=base_transform)
train_aug = datasets.CIFAR10(root="./data", train=True, download=True, transform=aug_transform)
test_ds = datasets.CIFAR10(root="./data", train=False, download=True, transform=base_transform)

train_base, _ = random_split(train_base, [20000, len(train_base)-20000])
train_aug, _ = random_split(train_aug, [20000, len(train_aug)-20000])

test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)
base_loader = DataLoader(train_base, batch_size=128, shuffle=True)
aug_loader = DataLoader(train_aug, batch_size=128, shuffle=True)

class SmallTorchCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128), nn.ReLU(),
            nn.Linear(128, 10)
        )
    def forward(self, x):
        return self.net(x)

def run_torch(loader, epochs=3):
    model = SmallTorchCNN().to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = crit(model(x), y)
            loss.backward()
            opt.step()
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total

torch_base_acc = run_torch(base_loader)
torch_aug_acc = run_torch(aug_loader)

print("PyTorch baseline accuracy:", round(torch_base_acc, 4))
print("PyTorch augmented accuracy:", round(torch_aug_acc, 4))